# Évaluation de la Robustesse (v2)

Ce notebook évalue la stabilité des performances et des explications face à des perturbations des données.

In [ ]:
import os
import sys
from pathlib import Path
import pandas as pd
import joblib
import importlib

# Robust path setup
project_root = Path(os.getcwd())
if project_root.name == 'notebooks':
    project_root = project_root.parent

src_path = str(project_root / "src")
if src_path not in sys.path:
    sys.path.append(src_path)

print(f"Racine du projet : {project_root}")

# Import common components
from xai_clinical.data.pancan_loader import PANCANLoader, BRCALoader, PANCANBRCAFusion
from xai_clinical.models.classifiers import ClassifierFactory
from xai_clinical.explainability.shap_explainer import SHAPExplainer


## 1. Robustesse au Bruit Gaussien

On observe la dégradation de l'AUC en fonction de l'écart-type du bruit ajouté aux features numériques.

In [ ]:
noise_levels = [0, 0.01, 0.05, 0.1, 0.2, 0.5]
results = []

for sigma in noise_levels:
    noise = np.random.normal(0, sigma, X_test.shape)
    X_noisy = X_test + noise
    y_pred = model.predict_proba(X_noisy)[:, 1]
    auc = roc_auc_score(y_test, y_pred)
    results.append({'Sigma': sigma, 'AUC': auc})

df_noise = pd.DataFrame(results)
plt.figure(figsize=(8, 5))
plt.plot(df_noise['Sigma'], df_noise['AUC'], marker='o')
plt.title("Impact du bruit gaussien sur l'AUC")
plt.xlabel("Écart-type du bruit")
plt.ylabel("AUC Test")
plt.grid(True)
plt.show()

## 2. Robustesse aux Données Manquantes

On simule l'absence de données en mettant des features à zéro (après scaling, zéro représente souvent la moyenne/médiane).

In [ ]:
missing_rates = [0, 0.1, 0.2, 0.3, 0.5]
results_missing = []

for rate in missing_rates:
    X_missing = X_test.copy()
    mask = np.random.rand(*X_missing.shape) < rate
    X_missing[mask] = 0
    y_pred = model.predict_proba(X_missing)[:, 1]
    auc = roc_auc_score(y_test, y_pred)
    results_missing.append({'Rate': rate, 'AUC': auc})

df_missing = pd.DataFrame(results_missing)
plt.figure(figsize=(8, 5))
plt.plot(df_missing['Rate'], df_missing['AUC'], marker='s', color='orange')
plt.title("Impact du taux de données manquantes sur l'AUC")
plt.xlabel("Taux d'omission")
plt.ylabel("AUC Test")
plt.grid(True)
plt.show()